# Part 3: The Training Loop

You have a model. Now you need to teach it language. The training loop is where the model actually learns, and every decision here affects whether your model converges or diverges into nonsense.

## The Training Objective

GPT is trained with **next-token prediction**: given tokens `[t0, t1, ..., tn]`, predict `[t1, t2, ..., tn+1]`. The loss function is cross-entropy between the model's predicted probability distribution and the actual next token.

This is a self-supervised task — the labels come from the data itself. Every piece of text is simultaneously input and target, just shifted by one position.

## Setup

In [ ]:
!pip install -q torch tqdm

# Download the Shakespeare dataset
import urllib.request, os
os.makedirs("data", exist_ok=True)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "data/shakespeare.txt"
)
print("Ready.")

## The Model (from Part 2)

Copy in the model architecture from Part 2.

In [ ]:
import torch
import torch.nn as nn
from dataclasses import dataclass

@dataclass
class GPTConfig:
    vocab_size: int = 65
    block_size: int = 256
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 384

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.n_head = config.n_head
        self.n_embd = config.n_embd

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        head_dim = C // self.n_head
        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)
        y = torch.nn.functional.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)

    def forward(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(0, T, device=idx.device)
        x = self.transformer.wte(idx) + self.transformer.wpe(pos)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = nn.functional.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1)
            )
        return logits, loss

print("Model classes defined.")

## Step 1: Data Loading

Each batch consists of:
- `x`: characters from position `i` to `i + block_size` (input)
- `y`: characters from position `i+1` to `i + block_size + 1` (target — shifted by one)

The function also returns `stoi`/`itos` mappings — you'll need these for text generation.

In [ ]:
def load_data(filepath, block_size, batch_size, device):
    with open(filepath, "r") as f:
        text = f.read()

    chars = sorted(set(text))
    vocab_size = len(chars)
    stoi = {c: i for i, c in enumerate(chars)}
    itos = {i: c for c, i in stoi.items()}

    tokens = torch.tensor([stoi[c] for c in text], dtype=torch.long)
    print(f"Dataset: {len(tokens):,} chars, vocab size: {vocab_size}")

    def get_batch(split_tokens):
        ix = torch.randint(len(split_tokens) - block_size - 1, (batch_size,))
        x = torch.stack([split_tokens[i:i + block_size] for i in ix]).to(device)
        y = torch.stack([split_tokens[i + 1:i + block_size + 1] for i in ix]).to(device)
        return x, y

    n = int(0.9 * len(tokens))
    get_train = lambda: get_batch(tokens[:n])
    get_val = lambda: get_batch(tokens[n:])
    return get_train, get_val, vocab_size, stoi, itos

## Step 2: Device Setup

The training automatically uses the best available accelerator:
- **MPS**: Apple Silicon GPU (~2-3× speedup over CPU)
- **CUDA**: NVIDIA GPU (much faster — use this in Colab with GPU runtime)
- **CPU**: fallback for any hardware

In [ ]:
def get_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")     # Apple Silicon GPU
    elif torch.cuda.is_available():
        return torch.device("cuda")    # NVIDIA GPU
    return torch.device("cpu")

device = get_device()
print(f"Using device: {device}")

## Step 3: Learning Rate Schedule

Two phases:

1. **Warmup** (first ~100 steps): Ramp the learning rate from near-zero up to `max_lr`. This gives the optimizer time to calibrate its moment estimates before making large updates.

2. **Cosine decay** (remaining steps): Smoothly decrease the learning rate. Large updates early (explore), small updates late (refine).

In [ ]:
import math

def get_lr(step, warmup_steps, max_steps, max_lr, min_lr):
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    if step >= max_steps:
        return min_lr
    progress = (step - warmup_steps) / (max_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))

In [ ]:
# Visualize the learning rate schedule
import matplotlib.pyplot as plt

max_steps = 5000
warmup = 100
max_lr = 1e-3
min_lr = max_lr * 0.1

steps = list(range(max_steps))
lrs = [get_lr(s, warmup, max_steps, max_lr, min_lr) for s in steps]

plt.figure(figsize=(10, 4))
plt.plot(steps, lrs)
plt.axvline(warmup, color='orange', linestyle='--', label='End of warmup')
plt.xlabel("Step")
plt.ylabel("Learning Rate")
plt.title("Cosine LR Schedule with Linear Warmup")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 4: The Full Training Loop

Key components:
- **Validation loss**: every 100 steps, evaluate on held-out data
- **Gradient clipping**: caps total gradient magnitude at 1.0 — prevents large gradients from blowing up the weights
- **Sample generation**: every 100 steps, generate text so you can watch the model learn
- **Checkpointing**: save model state periodically (includes `stoi`/`itos` for generation without original data)

In [ ]:
# Simple generation function (full version in Part 4)
@torch.no_grad()
def generate(model, prompt, stoi, itos, max_new_tokens=100, temperature=0.8, top_k=40):
    device = next(model.parameters()).device
    tokens = [stoi[c] for c in prompt if c in stoi]
    idx = torch.tensor([tokens], dtype=torch.long, device=device)
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.config.block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature
        if top_k > 0:
            values, _ = torch.topk(logits, top_k)
            logits[logits < values[:, -1:]] = float("-inf")
        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_token], dim=1)
    return "".join([itos[i] for i in idx[0].tolist()])

In [ ]:
import json
from tqdm.notebook import tqdm

def train(data_path, max_steps=5000, batch_size=64,
          n_layer=6, n_head=6, n_embd=384, block_size=256):
    device = get_device()
    print(f"Using device: {device}")

    get_train_batch, get_val_batch, vocab_size, stoi, itos = load_data(
        data_path, block_size, batch_size, device
    )

    config = GPTConfig(
        vocab_size=vocab_size,
        block_size=block_size,
        n_layer=n_layer,
        n_head=n_head,
        n_embd=n_embd,
    )
    model = GPT(config).to(device)
    print(f"Model: {n_layer}L/{n_head}H/{n_embd}D, "
          f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params")

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

    max_lr = 1e-3
    min_lr = max_lr * 0.1
    warmup_steps = 100

    loss_log = {"steps": [], "train": [], "val_steps": [], "val": []}

    pbar = tqdm(range(max_steps), desc="Training")
    for step in pbar:
        # --- validation loss ---
        if step % 100 == 0:
            model.eval()
            with torch.no_grad():
                val_losses = []
                for _ in range(20):
                    x, y = get_val_batch()
                    _, loss = model(x, y)
                    val_losses.append(loss.item())
                val_loss = sum(val_losses) / len(val_losses)
                tqdm.write(f"Step {step:5d} | val loss: {val_loss:.4f}")
                loss_log["val_steps"].append(step)
                loss_log["val"].append(val_loss)
            model.train()

        # --- update learning rate ---
        lr = get_lr(step, warmup_steps, max_steps, max_lr, min_lr)
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr

        # --- training step ---
        x, y = get_train_batch()
        _, loss = model(x, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{lr:.2e}")

        loss_log["steps"].append(step)
        loss_log["train"].append(loss.item())

        # --- generate sample ---
        if step > 0 and step % 500 == 0:
            model.eval()
            sample = generate(model, "To be or not", stoi, itos,
                              max_new_tokens=100, temperature=0.8)
            tqdm.write(f"\n--- Step {step} sample ---\n{sample}\n---\n")
            model.train()

        # --- save checkpoint ---
        if step > 0 and step % 1000 == 0:
            torch.save({
                "step": step,
                "model_state_dict": model.state_dict(),
                "config": config,
                "stoi": stoi,
                "itos": itos,
            }, f"checkpoint_{step}.pt")
            tqdm.write(f"Saved checkpoint_{step}.pt")

    # --- save final checkpoint and loss log ---
    torch.save({
        "step": max_steps,
        "model_state_dict": model.state_dict(),
        "config": config,
        "stoi": stoi,
        "itos": itos,
    }, "checkpoint_final.pt")

    with open("loss_log.json", "w") as f:
        json.dump(loss_log, f)

    print("\nTraining complete. Saved checkpoint_final.pt and loss_log.json")
    return model, stoi, itos, loss_log

## What Loss Numbers Mean (Character-Level, vocab=65)

| Loss | Meaning |
|------|---------|
| ~4.2 | Random (untrained). `ln(65) ≈ 4.17` |
| ~3.3 | Learned character frequencies (which letters are common) |
| ~2.5 | Learned common bigrams ("th", "he", "in") |
| ~1.5–2.0 | Generates recognizable words and Shakespeare-like structure |
| ~1.0–1.2 | Good quality — generates verse with character names, line breaks |
| <1.0 | Likely memorizing the training data |

## Run Training

The cell below runs training with the default Medium config (6L/6H/384D, 5000 steps).

**On Colab with a GPU (T4)**: ~10-15 minutes  
**On CPU**: very slow — try the Tiny config instead: `n_layer=2, n_head=2, n_embd=128`

In [ ]:
# Tip: if running on CPU, use the Tiny config:
# model, stoi, itos, loss_log = train("data/shakespeare.txt", n_layer=2, n_head=2, n_embd=128, max_steps=2000)

model, stoi, itos, loss_log = train("data/shakespeare.txt")

## Overfitting: Train Loss vs Val Loss

With 10M parameters and only ~1M characters of Shakespeare, the model will **overfit** — it memorizes the training data instead of learning general patterns. The **best model** is the one with the lowest validation loss, not the final checkpoint.

In [ ]:
import matplotlib.pyplot as plt

with open("loss_log.json") as f:
    log = json.load(f)

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(log["steps"], log["train"], alpha=0.3, label="Train loss")
ax.plot(log["val_steps"], log["val"], color="red", linewidth=2, label="Val loss")

# Mark the best validation checkpoint
best_val_idx = log["val"].index(min(log["val"]))
best_step = log["val_steps"][best_val_idx]
best_val = log["val"][best_val_idx]
ax.axvline(best_step, color='green', linestyle='--', label=f'Best val loss ({best_val:.3f}) at step {best_step}')

ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("Training vs Validation Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Best validation loss: {best_val:.4f} at step {best_step}")
print(f"Final validation loss: {log['val'][-1]:.4f}")

## Key Takeaways

- The objective is next-character prediction with cross-entropy loss
- AdamW with `lr=1e-3` and cosine decay is a good starting point
- Gradient clipping at 1.0 prevents training instability
- Generate samples during training — it's the best way to see progress
- **Watch the gap between train and val loss** — when val loss starts rising, you're overfitting
- The best model isn't the one with the lowest train loss — it's the one with the lowest val loss

**Next: [Part 4 — Text Generation](04-text-generation.ipynb)**